# Labwork 5 — Nonlinear least squares: Gauss–Newton and Levenberg–Marquardt

**Week 2 · Day 5 · ≈ 170 min at the keyboard**

Read Lecture 5 first. This labwork fills in part of the `optlab` package you cloned;
you edit the real source files, and the notebook checks your work as you go.

**What you build today:** the two workhorses of curve fitting, both reusing yesterday's Cholesky untouched

**Files you will open:**

- `problems/curve_fitting.py`
- `optimizers/least_squares.py`

> **The rule.** `src/optlab/interfaces.py`, `results.py`, `errors.py` and `types.py` are
> **provided** — never edit them. Everything else under `src/optlab/` is yours: replace
> each `raise NotImplementedError` with working code, keeping the signature and honouring
> the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone and gives you a
`check()` helper that runs a specific test file and reports what happened.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
print("optlab:", OPTLAB)


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

---

## Exercise 1 — Residuals and Jacobians  *(≈ 45 min)*

Implement `ExpDecay` and `GaussianPeak` in `curve_fitting.py`: `residuals(x)` is
`model(t; x) − y`, and `jacobian(x)` is the analytic `∂r/∂x` of shape `(m, n)`.

**Validate `jacobian` against `numerical_jacobian` before you let any optimizer near it.**
An analytic Jacobian with a sign error does not raise — it just makes the optimizer
mysteriously fail, and you will lose an hour.

Also check `g = Jᵀr` against `check_gradient` on `½‖r‖²`. Both day-1 tools, still earning
their keep.

Notice what these classes are *not*: they implement `LeastSquaresProblem`, not `Objective`.
Collapsing `r` and `J` into `value`/`gradient` would throw away the exact structure the
next two exercises exploit.

**Open:** `src/optlab/problems/curve_fitting.py`

In [ ]:
edit("problems/curve_fitting.py")
check("-m", "day5")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

For `y = a·exp(−bt) + c`, the three columns of `J` are `exp(−bt)`, `−a·t·exp(−bt)` and `1`.

</details>

---

## Exercise 2 — Gauss–Newton, and making it fail  *(≈ 45 min)*

Implement `GaussNewton.minimize`: solve `(JᵀJ)δ = −Jᵀr` with the **day-4
`CholeskySolver`**, injected — reused, not rewritten — then `x += δ`.

Two runs on `ExpDecay` with `(a, b, c) = (2.5, 1.3, 0.5)`:

1. **Good start `(1, 1, 1)`** — converges in a handful of iterations, and on noiseless data
   recovers the parameters to `1e-8`.
2. **Hard start `(1, 10, 1)`** — past `t ≈ 0.5` the model is nearly constant, so `a` and
   `b` are barely distinguishable, `JᵀJ` is near-singular, and the first step overshoots to
   an astronomical cost.

**Write a test that asserts the second one fails, and leave it in.** It is not a defect to
be patched; it is the motivation for Exercise 3, and Levenberg–Marquardt is unintelligible
until you have watched what it repairs.

**Open:** `src/optlab/optimizers/least_squares.py`

In [ ]:
check("-m", "day5")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

Print `np.linalg.cond(J.T @ J)` at both starting points. The ratio between them is the whole story.

</details>

---

## Exercise 3 — Levenberg–Marquardt  *(≈ 70 min)*

Implement `LevenbergMarquardt.minimize`: solve `(JᵀJ + λI)δ = −Jᵀr`, then decide
whether to keep the step using the **gain ratio**

```
rho = (cost(x) - cost(x + delta)) / predicted,    predicted = 0.5 * delta @ (lam * delta - g)
```

`rho > 0`: accept, and decrease `λ` — the model is trustworthy, be bolder.
`rho ≤ 0`: **reject the step**, and multiply `λ` by 10 — the model lied, shrink the region.

Note what you get for free: `JᵀJ ⪰ 0` always, so `JᵀJ + λI ≻ 0` for any `λ > 0`, and
**Cholesky can never fail here**. The exception path is unreachable by construction. That
is a direct payoff of having written the solver against an interface.

Verify: it converges from the hard start that destroyed Gauss–Newton; it recovers the true
parameters; and it matches `scipy.optimize.least_squares(method="lm")` (an oracle — tests
only, never `src/`).

Finally, the honest limit. Fit `y = a·sin(ωt + φ)` starting with `ω` far off. LM converges
— confidently, quickly, `converged=True` — to a **local** minimum. Judge by the cost, not
by the flag. A small gradient proves stationarity, never optimality.

**Open:** `src/optlab/optimizers/least_squares.py`

In [ ]:
check("-m", "day5")
check("-m", "contract")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

Clamp the printed `rho` when you trace it: a rejected first step can give a ratio of `-1e70`, which is correct and unreadable.

</details>

---

## Checkpoint

Everything from day 1 to day 5 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1 or day2 or day3 or day4 or day5")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

---

## Before the debrief

Be able to answer:

1. Which term does Gauss–Newton drop from the exact Hessian, and when is dropping it
   dangerous?
2. Why is LM naturally compatible with Cholesky while Gauss–Newton is not?
3. How does the gain ratio decide whether to trust the model more or less?
4. What is the difference between a line search and a trust region?